[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nbrg-ppcu/prokbert/blob/development/examples/sequence_dataset.ipynb)

# Working with sequence and embedding datasets

This notebook shows how to store many contigs as **one concatenated sequence plus metadata**, and how to look up sequences and embeddings by contig and coordinate. It has 4 steps:

1. Loading contigs from a FASTA file
2. Building, saving and loading a `SequenceDataset`
3. Querying sequences by ID, coordinate and orientation
4. Looking up embeddings with an `EmbeddingDataset`

We use a tiny dataset: the first contigs of the extremophile phage dataset, truncated to 20–30 nucleotides, so every result can be checked manually.

## Setting up the environment

The examples run on a CPU, so no GPU is needed. Install ProkBERT from GitHub:

In [1]:
# !pip install git+https://github.com/nbrg-ppcu/prokbert.git@development --quiet

In [2]:
import os
import yaml
import logging
import requests

import torch
import pandas as pd

from prokbert import helper
from prokbert.sequence_dataset import SequenceDataset, EmbeddingDataset

In [3]:
logging.basicConfig(level=logging.DEBUG, format="%(message)s")

## Preparing the dataset

You can either use your own FASTA file (set `DATA_PATH` to it) or use the small test file from the ProkBERT repository. The code below uses the local file if it exists, and downloads it otherwise (for example in Colab).

The file contains 3 *Aeromonas* phage contigs, truncated to 30, 25 and 20 nucleotides.

In [4]:

DATA_PATH = "/home/osvathm/prokbert/tests/data/extremophiles_mini.fasta"
DATA_URL = (
    "https://raw.githubusercontent.com/nbrg-ppcu/prokbert/development/"
    "tests/data/extremophiles_mini.fasta"
)

if not os.path.exists(DATA_PATH):
    os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)
    response = requests.get(DATA_URL, timeout=30)
    response.raise_for_status()
    with open(DATA_PATH, "w") as f:
        f.write(response.text)

# print(open(DATA_PATH).read())

## 1. Loading contigs

`load_contigs` reads one or more FASTA files (plain or `.gz`) and returns one record per contig, with its ID, sequence (uppercase), orientation and description.

`convert_to` turns the records into a pandas `DataFrame` or a Hugging Face `Dataset`, which is handy for inspection.

In [5]:
seq_ds = SequenceDataset()

In [6]:
contigs = seq_ds.load_contig(DATA_PATH)
# same as above just with multiple files
# contigs = seq_ds.load_contigs([DATA_PATH, DATA_PATH])
helper.convert_to(contigs, return_as="pandas")

,genome_id,contig_id,sequence,orientation,description
0,None,OM913597.1,TCTCGACCCGTCCCACCGCAATCGCCCCCA,forward,"OM913597.1 Aeromonas phage vB_AspA_Bolek, synt..."
1,None,OM913598.1,TCTCGACCCGCCCCACCGCAACCT,forward,"OM913598.1 Aeromonas phage vB_AspA_Lolek, synt..."
2,None,OM913599.1,CTCGCCTCGCCTCGCTTTCA,forward,"OM913599.1 Aeromonas phage vB_AspA_Tola, synth..."
3,None,OQ832096.1,ACGTNACGTNGGATCCTTAAA,forward,"OQ832096.1 Halomonas phage vB_HmeY_H4907, synt..."
4,None,JX507079.1,GATTACTCGCCTCGCCTCGC,forward,"JX507079.1 Acidithiobacillus phage AcaML1, syn..."


## 2. Building the dataset

`create_dataset` concatenates all contigs into **one long string** (`seq_ds.sequence`) and stores, for each contig, where it starts and ends in that string (`coordinate`, a half-open interval `[start, end)`, like Python slicing).

Why one string? It makes position-based operations simple: sampling a random window from the whole dataset, memory-mapping the sequence from disk, or finding which contig a position belongs to.

In [7]:
seq_ds.create_dataset([DATA_PATH])

In [8]:
print("Total length:", len(seq_ds.sequence))
print(seq_ds.sequence)

Total length: 115
TCTCGACCCGTCCCACCGCAATCGCCCCCATCTCGACCCGCCCCACCGCAACCTCTCGCCTCGCCTCGCTTTCAACGTNACGTNGGATCCTTAAAGATTACTCGCCTCGCCTCGC


In [9]:
pd.DataFrame(seq_ds.metadata)

,genome_id,contig_id,sequence_id,coordinate,orientation,description
0,None,OM913597.1,0,"(0, 30)",forward,"OM913597.1 Aeromonas phage vB_AspA_Bolek, synt..."
1,None,OM913598.1,1,"(30, 54)",forward,"OM913598.1 Aeromonas phage vB_AspA_Lolek, synt..."
2,None,OM913599.1,2,"(54, 74)",forward,"OM913599.1 Aeromonas phage vB_AspA_Tola, synth..."
3,None,OQ832096.1,3,"(74, 95)",forward,"OQ832096.1 Halomonas phage vB_HmeY_H4907, synt..."
4,None,JX507079.1,4,"(95, 115)",forward,"JX507079.1 Acidithiobacillus phage AcaML1, syn..."


Each contig's slice of the concatenated sequence gives back exactly the original contig:

In [10]:
for meta in seq_ds.metadata:
    start, end = meta["coordinate"]
    print(f"{meta['contig_id']:12} [{start:3}, {end:3})  {seq_ds.sequence[start:end]}")

OM913597.1   [  0,  30)  TCTCGACCCGTCCCACCGCAATCGCCCCCA
OM913598.1   [ 30,  54)  TCTCGACCCGCCCCACCGCAACCT
OM913599.1   [ 54,  74)  CTCGCCTCGCCTCGCTTTCA
OQ832096.1   [ 74,  95)  ACGTNACGTNGGATCCTTAAA
JX507079.1   [ 95, 115)  GATTACTCGCCTCGCCTCGC


### Saving and loading

The sequence is saved as a NumPy array (1 byte per nucleotide), and the metadata as JSON. Loading them back gives the same dataset without parsing the FASTA again.

In [11]:
SAVE_DIR = "outputs/toy_example"
seq_ds.create_dataset([DATA_PATH], save_dir=SAVE_DIR)

loaded_ds = SequenceDataset()
loaded_ds.load_dataset(SAVE_DIR, to_string=True)

assert loaded_ds.sequence == seq_ds.sequence

Dataset saved to 'outputs/toy_example'.


In [12]:
loaded_ds = SequenceDataset()
loaded_ds.load_dataset(SAVE_DIR, to_string=False)

assert loaded_ds.sequence.tobytes().decode("ascii") == seq_ds.sequence

print(os.listdir(SAVE_DIR))

['config.yaml', 'embedding_forward.pt', 'sequence.npy', 'metadata.json']


## 3. Querying sequences

### From sequence ID to coordinates, and back

`get_coordinates_from_sequence_id` returns where a contig lies in the concatenated sequence.

`get_sequence_id_from_start_coordinate` does the opposite: given a position, it finds the contig containing it with a binary search, in O(log n) time.

In [13]:
print(seq_ds.get_coordinates_from_sequence_id(1))        # (30, 55)

for position in [0, 29, 30, 54, 55, 74]:
    print(position, "->", seq_ds.get_sequence_id_from_start_coordinate(position))

(30, 54)
0 -> 0
29 -> 0
30 -> 1
54 -> 2
55 -> 2
74 -> 3


### Extracting a region

`get_sequence_from_metadata` takes coordinates **relative to the contig** (not to the concatenated sequence) and an orientation:

- `forward`: the sequence as stored
- `backward`: its reverse complement (complement each base, then reverse)

Try changing the IDs and coordinates!

In [14]:
fwd = seq_ds.get_sequence_from_metadata(0, 0, 10, "forward")
bwd = seq_ds.get_sequence_from_metadata(0, 0, 10, "backward")

print("forward :", fwd)   # TCTCGACCCG
print("backward:", bwd)   # CGGGTCGAGA

assert bwd == seq_ds.reverse_complement(fwd)

forward : TCTCGACCCG
backward: CGGGTCGAGA


Asking for coordinates outside the contig raises an error instead of silently returning bases from another contig:

In [15]:
try:
    seq_ds.get_sequence_from_metadata(
        sequence_id=2, # contig 2 is only 20 nt long
        start_coor=0,
        end_coor=50,
        orientation="forward"
    )
except ValueError as e:
    print("ValueError:", e)

ValueError: Coordinates 0-50 are out of bounds for sequence ID 2 with sequence length 20. (The coordinates were mapped to 54-104 with full sequence length 115.)


## 4. Looking up embeddings

A language model like ProkBERT does not see nucleotides directly but **tokens**. With LCA tokenization, k-mers of size `k` are taken at every `shift`-th position, so a sequence of length `L` gives

$$N = \left\lfloor \frac{L - k}{s} \right\rfloor + 1$$

k-mers, plus the special tokens `[CLS]` and `[SEP]`.

For example, `AAGTCCAGGATC` with `k = 6`, `shift = 2` gives `AAGTCC`, `GTCCAG`, `CCAGGA`, `AGGATC`.

In [16]:
print(EmbeddingDataset.calculate_lca_num_tokens(12, 6, 2, special_tokens=0))  # 4
print(EmbeddingDataset.calculate_lca_num_tokens(12, 6, 2))                    # 6, with [CLS] and [SEP]

4
6


### A toy embedding

Real embeddings come from running the model. Here we create a **fake** embedding with one row per token of the concatenated sequence, where every value in row `i` equals `i`. This way we can see exactly which rows a lookup returns.

The config describes how the embedding was produced (k-mer, shift, pooling, special tokens).

In [17]:
# trivial case
KMER, SHIFT = 6, 2
POOLING_LENGTH = 5

In [18]:
config = {
    "dataset": "extremophiles_mini",
    "tokenizer": "LCATokenizer",
    "kmer": KMER,
    "shift": SHIFT,
    "model": "neuralbioinfo/prokbert-mini-long",
    "pooling_strategy": "mean",
    "pooling_length": POOLING_LENGTH,
    "special_tokens": [],
    # "special_tokens": ["[CLS]", "[SEP]"], # example
}

SAVE_DIR = "outputs/toy_example"
os.makedirs(SAVE_DIR, exist_ok=True)

with open(f"{SAVE_DIR}/config.yaml", "w") as f:
    yaml.safe_dump(config, f)

hidden_dim = 8
pooling_len = POOLING_LENGTH
n_rows = EmbeddingDataset.calculate_lca_num_tokens(len(seq_ds.sequence), KMER, SHIFT, special_tokens=0)

embedding = torch.randn(n_rows, hidden_dim, dtype=torch.float32)  # (n_rows, hidden_dim)
print("Embedding shape:", embedding.shape)

embedding_pooled = embedding.reshape(-1, pooling_len, hidden_dim).mean(dim=1)
print("Pooled embedding shape:", embedding_pooled.shape)

torch.save(embedding_pooled, f"{SAVE_DIR}/embedding_forward.pt")

Embedding shape: torch.Size([55, 8])
Pooled embedding shape: torch.Size([11, 8])


The embeddings are loaded with **memory mapping**: rows are read from disk only when accessed, so even very large embedding files fit.

Now we ask for the embedding of a contig region, using contig-relative coordinates, just like with sequences:

In [19]:
emb_ds = EmbeddingDataset(
    sequence_dataset=seq_ds,
    config_path=f"{SAVE_DIR}/config.yaml",
    embedding_file=f"{SAVE_DIR}/embedding_forward.pt",
)

In [20]:
pd.DataFrame(seq_ds.metadata)

,genome_id,contig_id,sequence_id,coordinate,orientation,description
0,None,OM913597.1,0,"(0, 30)",forward,"OM913597.1 Aeromonas phage vB_AspA_Bolek, synt..."
1,None,OM913598.1,1,"(30, 54)",forward,"OM913598.1 Aeromonas phage vB_AspA_Lolek, synt..."
2,None,OM913599.1,2,"(54, 74)",forward,"OM913599.1 Aeromonas phage vB_AspA_Tola, synth..."
3,None,OQ832096.1,3,"(74, 95)",forward,"OQ832096.1 Halomonas phage vB_HmeY_H4907, synt..."
4,None,JX507079.1,4,"(95, 115)",forward,"JX507079.1 Acidithiobacillus phage AcaML1, syn..."


In [21]:
for sequence_id in range(len(seq_ds.metadata)):
    start, end = seq_ds.get_coordinates_from_sequence_id(sequence_id)

    emb = emb_ds.get_embedding_from_sequence_id_with_coordinates(
        sequence_id, 0, end - start
    )

Sequence ID 0 with requested coordinates 0-30.
Start token: 0, End token: 13 (in tokenised sequence with kmer=6 and shift=2)
Start index in embedding: 0, End index in embedding: 2 with embedding shape torch.Size([11, 8])
Sequence ID 1 with requested coordinates 0-24.
Start token: 13, End token: 25 (in tokenised sequence with kmer=6 and shift=2)
Start index in embedding: 2, End index in embedding: 5 with embedding shape torch.Size([11, 8])
Sequence ID 2 with requested coordinates 0-20.
Start token: 25, End token: 35 (in tokenised sequence with kmer=6 and shift=2)
Start index in embedding: 5, End index in embedding: 7 with embedding shape torch.Size([11, 8])
Sequence ID 3 with requested coordinates 0-21.
Start token: 35, End token: 45 (in tokenised sequence with kmer=6 and shift=2)
Start index in embedding: 7, End index in embedding: 9 with embedding shape torch.Size([11, 8])
Sequence ID 4 with requested coordinates 0-20.
Start token: 45, End token: 55 (in tokenised sequence with kmer=6 

In [22]:
# notes
# python and pytorch use 0-based, left-closed, right-open intervals [start, end[,
# i.e. seq[start:end] includes start but excludes end, and its length is end - start

# both ends are rounded down to whole pooling windows: the first window may start
# before start (possibly in the previous contig), and a partially covered last window
# is dropped (e.g. positions 50-54 of [30, 55) with pooling_length=5).
start = 30 // pooling_len
end = 54 // pooling_len
print(start, end)

6 10


## 5. Sampling segments

A language model can only read a limited number of tokens at once, so long contigs are cut into shorter **segments** before tokenization. The `Sampler` decides *where* the segments are, but doesn't read any sequence. It yields only coordinates, so it's cheap even for whole genomes.

Each segment has:

- `segment_id`: the segment's number
- `sequence_id`, `contig_id`, `genome_id`, `orientation`: which contig it comes from
- `coordinate`: `(start, end)` in the **concatenated** sequence, left-closed and right-open `[start, end)`, like Python slicing

Two parameters control the segment length:

- `max_length`: the longest allowed segment
- `min_segment_length`: shorter segments are dropped, because they carry too little context

There are two segmentation types: `contiguous` and `random`.

In [23]:
from prokbert.sampler import Sampler

In [24]:
# check metadata again
helper.convert_to(seq_ds.metadata)

,genome_id,contig_id,sequence_id,coordinate,orientation,description
0,None,OM913597.1,0,"(0, 30)",forward,"OM913597.1 Aeromonas phage vB_AspA_Bolek, synt..."
1,None,OM913598.1,1,"(30, 54)",forward,"OM913598.1 Aeromonas phage vB_AspA_Lolek, synt..."
2,None,OM913599.1,2,"(54, 74)",forward,"OM913599.1 Aeromonas phage vB_AspA_Tola, synth..."
3,None,OQ832096.1,3,"(74, 95)",forward,"OQ832096.1 Halomonas phage vB_HmeY_H4907, synt..."
4,None,JX507079.1,4,"(95, 115)",forward,"JX507079.1 Acidithiobacillus phage AcaML1, syn..."


### Contiguous segmentation

Each contig is cut into consecutive, disjunct chunks of `max_length`, starting at the beginning of the contig: `[0, 10)`, `[10, 20)`, … Every position is covered exactly once, except for a short remainder at the end of a contig, which is dropped if it is shorter than `min_segment_length`.

In [25]:
sampler = Sampler(sequence_dataset=seq_ds, min_segment_length=5, max_length=10, segmentation_type="contiguous")

for segment in iter(sampler): # iter over full dataset
    print(segment)

{'segment_id': 0, 'contig_id': 'OM913597.1', 'genome_id': None, 'sequence_id': 0, 'coordinate': (0, 10), 'orientation': 'forward'}
{'segment_id': 1, 'contig_id': 'OM913597.1', 'genome_id': None, 'sequence_id': 0, 'coordinate': (10, 20), 'orientation': 'forward'}
{'segment_id': 2, 'contig_id': 'OM913597.1', 'genome_id': None, 'sequence_id': 0, 'coordinate': (20, 30), 'orientation': 'forward'}
{'segment_id': 3, 'contig_id': 'OM913598.1', 'genome_id': None, 'sequence_id': 1, 'coordinate': (30, 40), 'orientation': 'forward'}
{'segment_id': 4, 'contig_id': 'OM913598.1', 'genome_id': None, 'sequence_id': 1, 'coordinate': (40, 50), 'orientation': 'forward'}
{'segment_id': 5, 'contig_id': 'OM913599.1', 'genome_id': None, 'sequence_id': 2, 'coordinate': (54, 64), 'orientation': 'forward'}
{'segment_id': 6, 'contig_id': 'OM913599.1', 'genome_id': None, 'sequence_id': 2, 'coordinate': (64, 74), 'orientation': 'forward'}
{'segment_id': 7, 'contig_id': 'OQ832096.1', 'genome_id': None, 'sequence_id'

### Random segmentation

The other option is random segmentation. It is an **infinite stream**: the sampler keeps producing new random segments for as long as you ask for them, so you decide how many to take (see the next cell). Each segment is made in two steps:

1. **Choose a contig.** A position is drawn uniformly from the whole concatenated sequence, and the contig containing it is chosen. This way, longer contigs are picked proportionally more often.
2. **Choose the segment inside the contig**, depending on the contig's length:
   - longer than `max_length`: a start position is drawn uniformly so that the whole segment fits inside the contig, and the segment is exactly `max_length` long;
   - between `min_segment_length` and `max_length`: the whole contig is returned as one segment;
   - shorter than `min_segment_length`: the contig is skipped, and a new position is drawn.

Segments never cross the boundary between two contigs, and their length is always between `min_segment_length` and `max_length`. Here `segment_id` is a running number that keeps increasing for as long as the sampler exists, so every random segment has a unique ID.

⚠️ Because the stream never ends, `list(sampler)` or a plain `for segment in sampler:` loop without a `break` would run forever. Always limit the number of segments, e.g. with `itertools.islice`.

Run the next cell several times: the segments change every time.

In [26]:
sampler = Sampler(sequence_dataset=seq_ds, min_segment_length=5, max_length=10, segmentation_type="random")

In [27]:
# %%time
for i, segment in enumerate(iter(sampler)):
    if i >= 15: break
    print(segment)

{'segment_id': 0, 'contig_id': 'OQ832096.1', 'genome_id': None, 'sequence_id': 3, 'coordinate': (77, 87), 'orientation': 'forward'}
{'segment_id': 1, 'contig_id': 'OM913599.1', 'genome_id': None, 'sequence_id': 2, 'coordinate': (62, 72), 'orientation': 'forward'}
{'segment_id': 2, 'contig_id': 'OQ832096.1', 'genome_id': None, 'sequence_id': 3, 'coordinate': (78, 88), 'orientation': 'forward'}
{'segment_id': 3, 'contig_id': 'OQ832096.1', 'genome_id': None, 'sequence_id': 3, 'coordinate': (78, 88), 'orientation': 'forward'}
{'segment_id': 4, 'contig_id': 'OM913598.1', 'genome_id': None, 'sequence_id': 1, 'coordinate': (30, 40), 'orientation': 'forward'}
{'segment_id': 5, 'contig_id': 'OM913597.1', 'genome_id': None, 'sequence_id': 0, 'coordinate': (12, 22), 'orientation': 'forward'}
{'segment_id': 6, 'contig_id': 'OM913599.1', 'genome_id': None, 'sequence_id': 2, 'coordinate': (61, 71), 'orientation': 'forward'}
{'segment_id': 7, 'contig_id': 'OQ832096.1', 'genome_id': None, 'sequence_id